In [1]:
import requests
import numpy as np
import json

API_URL = "https://isl-llms.ethz.ch/api"
API_KEY = 'fek2b44VS2RSgtaClXjBsbwSXg7pS65siKnNoYjmhn0CfxGthJWdWCmCINf-zG5eH1w'

# Tutorial to interact with the Watermark API using Python

You can visit https://isl-llms.ethz.ch/docs for an interactive documentation. Click "Authorize" to introduce your API-key and use the endpoints directly from the documentation.

Read the assignment for details on this task.

### Retrieve your watermarked token ids

Each student has a different set of tokenized strings that were watermarked with their secret watermarking keys. You can obtain yours by querying the endpoint `/tokens/` as follows.


In [2]:
def get_tokens():
    url_get_tokens = API_URL + "/tokens"

    payload = {}
    headers = {
      'x-api-key': API_KEY
    }

    return requests.request("GET", url_get_tokens, headers=headers, data=payload).json()

In [3]:
tokens = get_tokens()

In [4]:
# Get the token ids for each endpoint. Remember you must use them with the corect endpoint!
pvalue_token_ids = [token_entry for token_entry in tokens if token_entry["endpoint"]=="pvalue"]
bool_token_ids = [token_entry for token_entry in tokens if token_entry["endpoint"]=="bool"]

In [5]:
bool_token_ids[0]

{'tokens': '2324,15,187,50273,424,1785,1913,275,253,987,5680,27,38826,16973,285,23150,6019,329,2278,3863,689,5540,1107,3622,326,7943,247,9991,2190,9574,13,28056,313,10387,1203,1222,20556,390,21634,14,2388,282,24421,10,323,1896,20192,3268,8984,369,671,6312,281,7472,1880,10741,3882,6430,812,320,908,387,1029,16,46953,2308,327,9363,4918,534,310,4390,973,4232,407,690,1180,273,17085,15775,709,22,9318,14457,23,1184,496,436,1304,627,403,1643,6667,432,4102,9814,2175,7738,5678,2538,824,2439,16175,2390,533,5293,342,2074,13554,7668,3382,6973,4860,1491,417,1146,2130,5001,3167,31402,683,9190,21,62,42696,3951,51,25,1656,1723,841',
 'endpoint': 'bool'}

In [6]:
# You can obtain a list of integers by using the following function
def tokens_to_array(token_list_str: str) -> np.ndarray:
    return np.array([int(token_text) for token_text in token_list_str.split(",")])

tokens_to_array(bool_token_ids[0]["tokens"])

array([ 2324,    15,   187, 50273,   424,  1785,  1913,   275,   253,
         987,  5680,    27, 38826, 16973,   285, 23150,  6019,   329,
        2278,  3863,   689,  5540,  1107,  3622,   326,  7943,   247,
        9991,  2190,  9574,    13, 28056,   313, 10387,  1203,  1222,
       20556,   390, 21634,    14,  2388,   282, 24421,    10,   323,
        1896, 20192,  3268,  8984,   369,   671,  6312,   281,  7472,
        1880, 10741,  3882,  6430,   812,   320,   908,   387,  1029,
          16, 46953,  2308,   327,  9363,  4918,   534,   310,  4390,
         973,  4232,   407,   690,  1180,   273, 17085, 15775,   709,
          22,  9318, 14457,    23,  1184,   496,   436,  1304,   627,
         403,  1643,  6667,   432,  4102,  9814,  2175,  7738,  5678,
        2538,   824,  2439, 16175,  2390,   533,  5293,   342,  2074,
       13554,  7668,  3382,  6973,  4860,  1491,   417,  1146,  2130,
        5001,  3167, 31402,   683,  9190,    21,    62, 42696,  3951,
          51,    25,

### Check if a specific list of tokens is watermarked with your keys


#### p-value API


In [7]:
# Use this function to transform a list of integers into a valid string you can submit through the API
def tokens_to_string(token_list: np.array):
    return ",".join([str(token) for token in token_list])

tokens_to_string(tokens_to_array(bool_token_ids[0]["tokens"]))

'2324,15,187,50273,424,1785,1913,275,253,987,5680,27,38826,16973,285,23150,6019,329,2278,3863,689,5540,1107,3622,326,7943,247,9991,2190,9574,13,28056,313,10387,1203,1222,20556,390,21634,14,2388,282,24421,10,323,1896,20192,3268,8984,369,671,6312,281,7472,1880,10741,3882,6430,812,320,908,387,1029,16,46953,2308,327,9363,4918,534,310,4390,973,4232,407,690,1180,273,17085,15775,709,22,9318,14457,23,1184,496,436,1304,627,403,1643,6667,432,4102,9814,2175,7738,5678,2538,824,2439,16175,2390,533,5293,342,2074,13554,7668,3382,6973,4860,1491,417,1146,2130,5001,3167,31402,683,9190,21,62,42696,3951,51,25,1656,1723,841'

In [8]:
assert tokens_to_string(tokens_to_array(bool_token_ids[0]["tokens"])) == (bool_token_ids[0]["tokens"])

In [9]:
# Use this function to validate your tokens are valid
def validate_tokens(token_list_str: str):
    assert isinstance(token_list_str, str), "Your list must be a string of comma-separated integers"
    assert len(token_list_str)>0, "Your list is empty"

    # Split the string by commas
    parts = token_list_str.split(',')

    # Check if any part is empty or not an integer
    for part in parts:
        assert part.isdigit(), "Some entries in your list are not integers or are empty"

    assert not (len(parts) < 40 or len(parts) > 1000), "Your list must contain at least 40 and at most 1000 tokens. It is {} tokens long".format(len(parts))

In [10]:
def get_pvalue(tokens: str):
    validate_tokens(tokens)

    url_get_pvalue = API_URL + "/watermark/get_pvalue"

    payload = json.dumps({
      "tokens": tokens
    })

    headers = {
      'Content-Type': 'application/json',
      'x-api-key': API_KEY
    }

    response = requests.request("POST", url_get_pvalue, headers=headers, data=payload)

    return response.json()

In [ ]:
# get_pvalue(pvalue_token_ids[0]["tokens"])

#### boolean API


In [12]:
def get_bool(tokens: str):
    validate_tokens(tokens)

    url_get_bool = API_URL + "/watermark/get_bool"

    payload = json.dumps({
      "tokens": tokens
    })

    headers = {
      'Content-Type': 'application/json',
      'x-api-key': API_KEY
    }

    response = requests.request("POST", url_get_bool, headers=headers, data=payload)

    return response.json()

In [ ]:
# get_bool(bool_token_ids[0]["tokens"])

### Check your quota


In [14]:
url = "https://isl-llms.ethz.ch/check-limits/{}".format(API_KEY)

payload = {}
headers = {}

response = requests.request("GET", url, headers=headers, data=payload)

print(response.text)

{"user":"gsolun@student.ethz.ch","bool_left_today":3000,"pvalue_left_today":3000,"bool_left_total":11334,"pvalue_left_total":11125,"bool_day_reset_at":"window not started — next request will start a fresh 24h window","pvalue_day_reset_at":"window not started — next request will start a fresh 24h window"}


# Breaking the watermark (attack)

Detection: The API counts the unique consecutive token pairs `(s_{t-1}, s_t)` (duplicates skipped). With `red` = #unique pairs whose 2nd token is in the red list seeded by the 1st token and `total` = #unique pairs,

```
z = (red - total/2) / sqrt(total * 0.5 * 0.5)
p = 1 - norm.cdf(-z) # == norm.cdf(z)
watermarked <=> p < 0.01 <=> z < norm.ppf(0.01) ~= -2.326
```

Attack: Append a suffix whose every internal pair `(c_i, c_{i+1})` is red. Appending `L` tokens adds `L` new unique pairs (the boundary pair `(last_string_token, c_1)` plus `L-1` internal pairs). If the internal pairs are red, `2*red - total` moves from `-G` to `-G + L - 2 + 2b`, where `G = total - 2*red` is the string's green-excess and `b in {0,1}` is whether the boundary pair is red. So

```
minimal L  ~=  G - 2.326*sqrt(total + L) + (2 - 2b).
```

Because the watermark key is fixed within a subtask, a red pair is red for every string, so one red chain is a universal suffix. We take suffix tokens from ids in no string, so the suffix never reuses an existing pair.

This means: the shortest possible suffix is fixed by your strings' green-excess `G` (reported below). The two levers we control are making every internal pair red (optimal, 1 red pair per token is the theoretic max) and the boundary
pair: choosing `c_1` so `(last_token, c_1)` is red on the binding (largest-`G`) string saves 1-2 tokens. We do both.

Server gates: Every query must have >=50 tokens AND >=50 distinct consecutive pairs ("too repetitive" otherwise). So all probes below are all-distinct-token paths.

Red-pair discovery per endpoint:

- p-value: append to a 52-distinct-token filler (keeps `p ~ 0.5`): pair `(a,b)` is red iff the p-value goes up. We also recover each string's exact `red`/`total`/`G`.
- boolean: only 0/1, so we anchor at the threshold with a reusable balanced base `B`: a 50-distinct-pair path with exactly 17 reds (`R*(51)=R*(52)=17`). Then `B+[a]` (51 pairs) is `bool=1` iff `(B_end,a)` is green (certifies "green-connected" candidates), and for those, `B+[a,b]` (52 pairs) has `R=17+red(a,b)`, so `bool=0` iff `(a,b)` is red, one query certifies any edge.


In [ ]:
# Attack primitives
import time, random, datetime
from scipy.stats import norm

VOCAB = 50277 # valid token ids are [0, 50277)
MIN_TOKENS = 50 # server rule 1: ">= 50 tokens for watermark detection"
MIN_PAIRS = 50 # server rule 2: ">= 50 DISTINCT consecutive pairs" ("too repetitive" otherwise). This rule rules out any small-unique-pair probe. Every probe needs >= 50 distinct pairs, not just >= 50 tokens.
THRESHOLD_Z = norm.ppf(0.01) # ~ -2.3263
random.seed(0)

def max_red_watermarked(num_pairs):
    """Largest red-pair count R for which N pairs are still flagged watermarked
    (bool==1): bool==1 <=> z=(R-N/2)/sqrt(N/4) < ppf(0.01). R*(50)=16, R*(51)=R*(52)=17."""
    return max((red_count for red_count in range(num_pairs + 1) if (red_count - num_pairs / 2) / (num_pairs * 0.25) ** 0.5 < THRESHOLD_Z), default=-1)


# Logging: append the entire run to a timestamped log file 
LOG_PATH = "Q2_run.log"
log_file = open(LOG_PATH, "a")
def log(message, echo=True):
    """Append msg to Q2_run.log with a timestamp on every content line (flushed). Per-line stamping keeps any leading/embedded newline tidy (blank lines stay blank, content lines always carry a timestamp)."""
    timestamp = f"[{datetime.datetime.now().isoformat(timespec='seconds')}] "
    log_file.write("\n".join((timestamp + line_text) if line_text else "" for line_text in str(message).split("\n")) + "\n")
    log_file.flush()
    if echo:
        print(message)
log("================ run start ================", echo=False)


# Persistent query cache. Never re-spend quota on a repeated query. Every (endpoint, token-string) -> response is appended to a JSONL log and reloaded on startup. Re-running the notebook (after a crash / rate-limit / to tweak the build logic) replays cached answers for FREE and only sends queries that are genuinely new. The build is seeded (random.seed(0) below) so a clean re-run reissues the exact same probes -> 100% cache hits, 0 quota spent.
CACHE_PATH = "Q2_query_cache.jsonl"
query_cache = {}
api_stats = {"new_pvalue": 0, "new_bool": 0, "cache_hits": 0}

# Optional per-session live query caps. Cached responses do not count. Set either value to 100 (or another integer) before a live run if you want the notebook to stop, with cache/state already flushed, instead of spending more new queries in the current kernel session. Leave as None to run to completion.
MAX_NEW_PVALUE_QUERIES = 100
MAX_NEW_BOOL_QUERIES = 100

def query_limit(endpoint):
    return MAX_NEW_PVALUE_QUERIES if endpoint == "pvalue" else MAX_NEW_BOOL_QUERIES

def load_cache():
    import os
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH) as opened_file:
            for line in opened_file:
                line = line.strip()
                if not line:
                    continue
                try:
                    record = json.loads(line)
                    query_cache[(record["endpoint"], record["tokens"])] = record["resp"]
                except Exception:
                    pass
    log(f"query cache: loaded {len(query_cache)} stored results from {CACHE_PATH}")

load_cache()
cache_file = open(CACHE_PATH, "a") # append handle kept open for the session

def cache_put(endpoint, tokens_str, resp):
    query_cache[(endpoint, tokens_str)] = resp
    cache_file.write(json.dumps({"endpoint": endpoint, "tokens": tokens_str, "resp": resp}) + "\n")
    cache_file.flush()


# API wrappers: cached + logged, return the `wmark_response` value
def api_call(api_function, tokens_list):
    tokens_string = tokens_to_string(np.asarray(tokens_list, dtype=int))
    endpoint = "pvalue" if api_function is get_pvalue else "bool"
    num_tokens = len(tokens_list)
    num_pairs = len({(int(tokens_list[index - 1]), int(tokens_list[index])) for index in range(1, num_tokens)})
    hit = query_cache.get((endpoint, tokens_string))
    if hit is not None: # cached (note: 0/0.0 are valid, not misses)
        api_stats["cache_hits"] += 1
        log(f"q  cache {endpoint:6s} ntok={num_tokens:4d} npairs={num_pairs:4d} -> {hit}", echo=False)
        return hit

    limit = query_limit(endpoint)
    if limit is not None and api_stats["new_" + endpoint] >= int(limit):
        save_state()
        cache_file.flush()
        pairs_file.flush()
        log(f"STOP: reached {endpoint} live query cap ({limit}). "
            "Progress is saved in Q2_query_cache.jsonl, Q2_known_pairs.jsonl, and Q2_state.json.")
        raise RuntimeError(
            f"Stopped after {limit} new {endpoint} queries in this session."
            "Restart/rerun later to continue from saved cache/state, or raise the cap."
        )

    try:
        response = api_function(tokens_string)
        if isinstance(response, dict) and "wmark_response" in response:
            value = response["wmark_response"]
            cache_put(endpoint, tokens_string, value)
            api_stats["new_" + endpoint] += 1
            log(f"q  NEW#{api_stats['new_' + endpoint]:<4d} {endpoint:6s} "
                f"ntok={num_tokens:4d} npairs={num_pairs:4d} -> {value}", echo=False)
            return value
    except Exception as error:
        response = repr(error)

    save_state()
    cache_file.flush()
    pairs_file.flush()
    log(f"ERROR: {endpoint} API call failed. Progress is saved. Last response: {response}")
    raise RuntimeError(f"{endpoint} API call failed. Last response: {response}")

def p_value_query(tokens_list): return float(api_call(get_pvalue, tokens_list)) # p-value endpoint
def bool_query(tokens_list): return int(api_call(get_bool, tokens_list)) # boolean endpoint

def print_quota_usage():
    log(f"this session: {api_stats['new_pvalue']} new p-value + "
        f"{api_stats['new_bool']} new bool queries sent "
        f"{api_stats['cache_hits']} served from cache "
        f"({len(query_cache)} total stored).")


# Run checkpoint: Persist solver progress so phases resume after a crash The query cache above already makes every individual query free on a re-run. This additionally records finished artifacts (balanced base, candidates, the discovered suffixes) so a restarted run skips completed phases and never loses a found answer. Delete Q2_state.json to force a clean recompute.
STATE_PATH = "Q2_state.json"

def load_state():
    import os
    if os.path.exists(STATE_PATH):
        try:
            with open(STATE_PATH) as opened_file:
                return json.load(opened_file)
        except Exception:
            pass
    return {}

STATE = load_state()

def save_state():
    import os
    temp_path = STATE_PATH + ".tmp"
    with open(temp_path, "w") as opened_file:
        json.dump(STATE, opened_file)
    os.replace(temp_path, STATE_PATH)

def checkpoint(**updates):
    """Update STATE and persist atomically, called at fine granularity (after each attached candidate / each chain edge / each finished start) so a crash loses nothing and a resume continues from the exact point."""
    STATE.update(updates)
    save_state()

if STATE:
    log(f"checkpoint: resumed run state with keys {sorted(STATE)}")


# Known-pair store: Record every pair colour proved with certainty. Both endpoints reveal pair colours with 100% certainty (p-value: p goes up/down. bool: the balanced base B isolates a single edge, and B+[a] proves green in-edges (B_end,a)). We persist red and green facts, namespaced by endpoint (the two endpoints use different watermark keys, so a pair's colour differs between them). This lets the oracles answer a known pair with ZERO queries even across runs where the base B changed (the query cache is keyed by probe string and would miss those. This is keyed by the pair).
PAIRS_PATH = "Q2_known_pairs.jsonl"
KNOWN_PAIRS = {} # (endpoint, a, b) -> True (red) / False (green)

def load_pairs():
    import os
    if os.path.exists(PAIRS_PATH):
        with open(PAIRS_PATH) as opened_file:
            for line in opened_file:
                line = line.strip()
                if not line:
                    continue
                try:
                    record = json.loads(line)
                    KNOWN_PAIRS[(record["ep"], record["a"], record["b"])] = record["red"]
                except Exception:
                    pass
    num_red = sum(1 for is_red_value in KNOWN_PAIRS.values() if is_red_value)
    log(f"known pairs: loaded {len(KNOWN_PAIRS)} ({num_red} red, {len(KNOWN_PAIRS) - num_red} green) from {PAIRS_PATH}")

load_pairs()
pairs_file = open(PAIRS_PATH, "a")

def known_pair(endpoint, token_a, token_b):
    return KNOWN_PAIRS.get((endpoint, int(token_a), int(token_b))) # True / False / None

def mark_pair(endpoint, token_a, token_b, is_red):
    """Record a CERTAIN pair colour (idempotent, crash-safe append)."""
    key = (endpoint, int(token_a), int(token_b))
    if key in KNOWN_PAIRS:
        return
    KNOWN_PAIRS[key] = bool(is_red)
    pairs_file.write(json.dumps({"ep": endpoint, "a": int(token_a), "b": int(token_b), "red": bool(is_red)}) + "\n")
    pairs_file.flush()

def print_known_pairs():
    for endpoint in ("pvalue", "bool"):
        reds = sum(1 for (pair_endpoint, token_a, token_b), is_red_value in KNOWN_PAIRS.items() if pair_endpoint == endpoint and is_red_value)
        greens = sum(1 for (pair_endpoint, token_a, token_b), is_red_value in KNOWN_PAIRS.items() if pair_endpoint == endpoint and not is_red_value)
        log(f"  known {endpoint} pairs: {reds} red, {greens} green")


def unique_pairs(tokens):
    return {(int(tokens[index - 1]), int(tokens[index])) for index in range(1, len(tokens))}


# Free token pool: ids that appear in no string of either endpoint
all_string_tokens = set()
for token_entry in tokens:
    all_string_tokens |= set(int(token) for token in tokens_to_array(token_entry["tokens"]))
FREE = [token_id for token_id in range(VOCAB) if token_id not in all_string_tokens]
random.shuffle(FREE)
log(f"free token pool size = {len(FREE)} (excluded {len(all_string_tokens)} string tokens)")


# Exact red count of a sequence from its p-value (p == norm.cdf(z))
def recover_red_count(tokens_list):
    pvalue = p_value_query(tokens_list)
    total = len(unique_pairs(tokens_list))
    if total == 0:
        return 0, 0
    clipped_p = min(max(pvalue, 1e-15), 1 - 1e-15)
    z_score = norm.ppf(clipped_p)
    return int(round((z_score * total ** 0.5 + total) / 2.0)), total


def theoretical_min_length(green_excess, total_pairs, boundary_red):
    """Smallest suffix length flipping a string with green-excess G, total T and boundary-pair colour b (1=red). z_new = (L-2+2b-G)/sqrt(T+L) >= THRESHOLD_Z."""
    length = 1
    while not (length - 2 + 2 * boundary_red - green_excess >= THRESHOLD_Z * (total_pairs + length) ** 0.5):
        length += 1
        if length > VOCAB:
            break
    return length


# Chain building
def grading_check_lengths(endpoint, estimate=None, cap=140):
    """Lengths at which to test all 7 strings during the build. Taken from the assignment's score breakpoints (p-value 40/41/42/50, bool 41/42/43/50, both flatten past 50) plus the analytic minimum (when known) and a coarse cadence. This is FAR fewer probes than testing every few edges. The trim afterwards still recovers the exact minimal length, so this never changes the score. It only controls how many (string x suffix) queries the build spends."""
    points = {40, 41, 42, 50} if endpoint == "pvalue" else {41, 42, 43, 50}
    if estimate: # tight lower bound is known
        points |= set(range(max(1, estimate), estimate + 3)) # expect to pass at the analytic min
    points |= set(range(24, 50, 8)) # low anchors -> small minima stop early
    points |= set(range(50, cap + 1, 10)) # >50: score flat -> sparse
    points.add(cap)
    return sorted(length for length in points if 1 <= length <= cap)

# Only run the exact minimal trim (minimal_prefix) when the suffix length lands in the scoring slope where shorter actually means a better grade (p-value 42..50, bool 43..50). In <=41 / <=42 the trim cannot improve the grade, so we skip it and save those string queries. Set this False to always trim to the exact minimum.
TRIM_ONLY_IN_SCORING_SLOPE = True
def scoring_slope(endpoint):
    return (42, 50) if endpoint == "pvalue" else (43, 50)

def grow_red_chain(edge_is_red, candidates, start, strings, detect_fn, passes,
                   check_lengths=None, hard_cap=140, progress_key=None):
    """Greedily grow a simple path of red edges from start. Rejected candidates are kept (a green successor of one node may be red for the next). Test all strings only when the chain length hits one of check_lengths (the grading boundaries), and stop once they all pass. Returns (chain, flipped) where flipped is True iff a check confirmed all strings flip at len(chain).

    If progress_key is given, the partial chain is checkpointed to STATE after every accepted edge, and a matching partial (same start) is resumed, so an interrupted build continues from the exact token it stopped at."""
    chain = [start]
    if progress_key: # resume the in-progress chain
        partial = STATE.get(progress_key + "_partial")
        if partial and partial[0] == start:
            chain = list(partial)
            log(f"    resumed partial {progress_key} chain at length {len(chain)}", echo=False)
    remaining = [candidate for candidate in candidates if candidate not in chain]
    checkset = set(check_lengths or ())
    flipped = False
    while len(chain) < hard_cap and remaining:
        current = chain[-1]
        hit = None
        for index, next_candidate in enumerate(remaining):
            if edge_is_red(current, next_candidate):
                hit = index
                break
        if hit is None:
            break
        chain.append(remaining.pop(hit))
        if progress_key:
            checkpoint(**{progress_key + "_partial": chain}) # save after each edge
        if len(chain) in checkset: # check only at grading boundaries
            if all(passes(detect_fn(list(string) + chain)) for string in strings):
                flipped = True
                break
    return chain, flipped

def minimal_prefix(chain, strings, detect_fn, passes, high_cap=None):
    """Smallest prefix length L of chain that flips all strings. Tests the cap first: if chain[:hi_cap] doesn't already flip, returns None in one check (lets best_suffix cheaply reject a start that can't beat the best)."""
    top = len(chain) if high_cap is None else min(len(chain), high_cap)
    if top < 1 or not all(passes(detect_fn(list(string) + chain[:top])) for string in strings):
        return None
    low, high, best = 1, top, top
    while low <= high:
        mid = (low + high) // 2
        if all(passes(detect_fn(list(string) + chain[:mid])) for string in strings):
            best = mid
            high = mid - 1
        else:
            low = mid + 1
    return best

def best_suffix(edge_is_red, candidates, strings, detect_fn, passes, starts,
                check_lengths=None, trim_range=None, full_marks_at=None,
                hard_cap=140, progress_key=None):
    """Build a red chain from each start token and keep the globally shortest suffix.

    The exact-minimal trim (minimal_prefix) is run only when the built flip length falls in trim_range (the scoring slope) elsewhere shortening cannot improve the grade, so we keep the built length and skip the trim queries. Other savings: the build checks only at check_lengths (grading boundaries). Once a best length is known, later starts are built no longer than best+1, and once a length full_marks_at is reached we stop trying more starts.

    Successful starts are checkpointed per start. A resume reuses the chain and (in the trim range) re-runs minimal_prefix against the curret cache."""
    done = (STATE.get(progress_key + "_starts") if progress_key else None) or {}
    best_chain, best_len = None, None
    for start_index, start_token in enumerate(starts):
        cap = hard_cap if best_len is None else min(hard_cap, best_len + 1)
        if str(start_token) in done: # built on a prior run
            chain, flipped = done[str(start_token)]["chain"], True
        else:
            chain, flipped = grow_red_chain(edge_is_red, candidates, start_token, strings, detect_fn,
                                            passes, check_lengths=check_lengths, hard_cap=cap,
                                            progress_key=progress_key)
            if progress_key and flipped: # only checkpoint working starts
                done[str(start_token)] = {"chain": chain}
                checkpoint(**{progress_key + "_starts": done})
        if not flipped:
            log(f"    start {start_index + 1}/{len(starts)}: no flipping suffix within cap {cap}")
            continue
        built_length = len(chain)
        if trim_range and trim_range[0] <= built_length <= trim_range[1]:
            high_cap = None if best_len is None else best_len - 1 # only accept strictly shorter
            length = minimal_prefix(chain, strings, detect_fn, passes, high_cap=high_cap)
            note = f"trimmed -> {length}" if length is not None else "not shorter than best"
        else:  # plateau or low score tail: no trim
            length = built_length if (best_len is None or built_length < best_len) else None
            note = f"flips at {built_length} (trim skipped: outside scoring slope)"
        log(f"    start {start_index + 1}/{len(starts)}: built {built_length} -> {note}")
        if length is not None and (best_len is None or length < best_len):
            best_len, best_chain = length, chain[:length]
        if full_marks_at is not None and best_len is not None and best_len <= full_marks_at:
            log(f"    full marks at length {best_len} (<= {full_marks_at}), stopping start search")
            break
    return best_chain, best_len


# p-value red edge oracle: red(a,b) <=> p(filler+[a,b]) > p(filler+[a]). 
# filler = 52 distinct tokens => filler+[a] has 52 distinct pairs, filler+[a,b] 53. 
# Both clear both gates (>=50 tokens and >=50 distinct pairs).
FILLER_LEN = MIN_PAIRS + 2
def make_pvalue_oracle():
    filler = FREE[:FILLER_LEN]
    base_p_values = {}
    def is_red(token_a, token_b):
        known_color = known_pair("pvalue", token_a, token_b) # answer for free if already proved
        if known_color is not None:
            return known_color
        if token_a not in base_p_values:
            base_p_values[token_a] = p_value_query(filler + [token_a])
        is_red_result = p_value_query(filler + [token_a, token_b]) > base_p_values[token_a] # red <=> p strictly increases
        mark_pair("pvalue", token_a, token_b, is_red_result)
        return is_red_result
    return is_red, FREE[FILLER_LEN:]


# boolean red-edge oracle via a reusable balanced base B 
# The >=50-distinct-pair gate rules out tiny repeated cycles, so we anchor at the detection threshold with a full size base instead. B is a 50-distinct-pair path with exactly 17 reds (R*(51)=R*(52)=17). Then, appending tokens to B:
# B+[a] (51 pairs): R = 17 + red(B_end,a) -> bool==1 <=> (B_end,a) green. -> certifies "green-connected" candidates a (a has a green in-edge from B_end).
# B+[a,b] (52 pairs) for a green-connected: R = 17 + red(a,b) -> bool==0 <=> (a,b) red. One query certifies any edge, both directions. 
# Every probe is an all-distinct path of >=53 tokens / >=51 distinct pairs -> accepted.
def bootstrap_base(max_bases=8000):
    """Find a 50-pair path B (51 distinct tokens) with EXACTLY 17 reds, and some
    green-connected seeds. Confirmed by mixed bool over B+[a]: mixed <=> R(B)=17
    (R(B)<=16 => all 1 and R(B)>=18 => all 0), so this is certain."""
    free = [token for token in FREE]
    for unused in range(max_bases):
        base = random.sample(free, 51) # 51 tokens -> 50 distinct pairs
        if bool_query(base) == 1: # bool(B)==1 => R(B)<=16, want 17
            continue
        rest = [token for token in free if token not in base]
        sample_candidates = random.sample(rest, 6)
        results = [bool_query(base + [candidate]) for candidate in sample_candidates] # B+[c] = 51 pairs
        if 0 in results and 1 in results: # mixed => R(B)=17 (certain)
            for candidate, result in zip(sample_candidates, results):
                mark_pair("bool", base[-1], candidate, result == 0) # bool0 <=> (B_end,c) red
            green_connected_seed = [candidate for candidate, result in zip(sample_candidates, results) if result == 1]
            return base, green_connected_seed
    raise RuntimeError("could not bootstrap a balanced base (try rerunning)")

def bool_edge_oracle(base):
    """Red-edge oracle, rebuilt from a (possibly resumed) base B. Valid for `a`
    that is green-connected (B_end -> a green). We only ever call it for such a."""
    base_end = base[-1]
    def is_red(token_a, token_b):
        known_color = known_pair("bool", token_a, token_b) # answer for free if already proved
        if known_color is not None:
            return known_color
        is_red_result = bool_query(base + [token_a, token_b]) == 0 # 52 pairs, 0 <=> (a,b) red
        mark_pair("bool", token_a, token_b, is_red_result)
        return is_red_result
    return is_red

def is_green_connected(base, token):
    """True iff (B_end, a) is green, i.e. bl(B+[a])==1 (B has R=17, so 51-pair probe)."""
    known_color = known_pair("bool", base[-1], token)
    if known_color is not None:
        return not known_color # known red => not green-connected
    is_green = bool_query(base + [token]) == 1
    mark_pair("bool", base[-1], token, not is_green)
    return is_green

def make_bool_oracle(n_candidates=150, base=None, green_connected_seed=None):
    """Bootstrap (or resume) the base B, collect `n_candidates` green-connected nodes, and return (edge_is_red, candidates, B). The base is checkpointed as soon as it is found, and the candidate list is checkpointed incrementally, so a crash mid-bootstrap or mid-attach resumes without redoing finished work."""
    if base is None:
        base, green_connected_seed = bootstrap_base()
        checkpoint(bool_base=list(base)) # persist the (expensive) base at once
    candidates = list(green_connected_seed or [])
    for token in FREE:
        if len(candidates) >= n_candidates:
            break
        if token in base or token in candidates:
            continue
        if is_green_connected(base, token):
            candidates.append(token)
            checkpoint(bool_candidates=list(candidates)) # persist after each attach
    return bool_edge_oracle(base), candidates, base


query cache: loaded 1532 stored results from Q2_query_cache.jsonl
checkpoint: resumed run state with keys ['bool_base', 'bool_candidates', 'bool_suffix', 'bool_suffix_42', 'pvalue_suffix']
known pairs: loaded 1222 (608 red, 614 green) from Q2_known_pairs.jsonl
free token pool size = 49187 (excluded 1090 string tokens)


### Solve the p-value endpoint


In [ ]:
# Solve the p-value endpoint
p_value_strings = [tokens_to_array(token_entry["tokens"]).tolist() for token_entry in pvalue_token_ids]

log("==== p-value endpoint ====", echo=False)
if "pvalue_suffix" in STATE:                          # resume: phase already finished
    if STATE.pop("pv_partial", None) is not None or STATE.pop("pv_starts", None) is not None:
        save_state()                                  # drop any stale transient progress
    YOUR_PVALUE_SUFFIX_LIST = STATE["pvalue_suffix"]
    p_value_length = len(YOUR_PVALUE_SUFFIX_LIST)
    log(f"resumed p-value suffix from checkpoint (length {p_value_length})")
else:
    p_value_is_red, p_value_candidate_pool = make_pvalue_oracle()

    # (a) exact green excess of each raw string + theoretical minimal suffix length.
    log("raw p-value strings:")
    p_value_meta = []
    for string in p_value_strings:
        red_count, total_pairs = recover_red_count(string)
        green_excess = total_pairs - 2 * red_count
        p_value_meta.append((string, red_count, total_pairs, green_excess, int(string[-1])))
        log(f"  red={red_count:3d}  total={total_pairs:3d}  green_excess={green_excess:3d}  min L (boundary red)={theoretical_min_length(green_excess, total_pairs, 1)}")
    worst_min_length = max(theoretical_min_length(green_excess, total_pairs, 1) for unused, red_count, total_pairs, green_excess, end_token in p_value_meta)
    log(f"==> best achievable p-value suffix ~ {worst_min_length} tokens  "
        f"({'<=41 reachable' if worst_min_length <= 41 else 'G too high for <=41 -> aim for partial credit'})")

    # (b) Boundary optimisation. Making the boundary pair (last_token, c_1) red saves 2 tokens on a string. The binding set = strings whose b=0 requirement is within 2 of the max (any of them could become the limiter once boundaries flip). Find c_1 red after all of them.
    requirements = [(theoretical_min_length(green_excess, total_pairs, 0), end_token) for unused, red_count, total_pairs, green_excess, end_token in p_value_meta]
    max_requirement = max(requirement for requirement, end_token in requirements)
    binding_ends = [end_token for requirement, end_token in requirements if requirement >= max_requirement - 2]
    log(f"binding strings: {len(binding_ends)} (optimizing the boundary pair on these)")

    starts, best_candidate, best_coverage = [], p_value_candidate_pool[0], -1
    for candidate in p_value_candidate_pool[:500]:
        coverage = 0
        for end_token in binding_ends: # short-circuit: stop at first green
            if p_value_is_red(end_token, candidate):
                coverage += 1
            else:
                break
        if coverage > best_coverage:
            best_coverage, best_candidate = coverage, candidate
        if coverage == len(binding_ends): # boundary red for every binding string
            starts.append(candidate)
            if len(starts) >= 3:
                break
    if not starts: # best partial coverage (never a random b=0)
        starts = [best_candidate]
        log(f"WARNING: boundary only partially optimized ({best_coverage}/{len(binding_ends)})")
    log(f"using {len(starts)} boundary-optimized start token(s) (full coverage of {len(binding_ends)} binding ends)")

    p_value_candidates = [candidate for candidate in p_value_candidate_pool if candidate not in starts]
    YOUR_PVALUE_SUFFIX_LIST, p_value_length = best_suffix(
        p_value_is_red, p_value_candidates, p_value_strings, p_value_query, lambda pvalue: pvalue >= 0.01, starts=starts,
        check_lengths=grading_check_lengths("pvalue", worst_min_length),
        trim_range=(scoring_slope("pvalue") if TRIM_ONLY_IN_SCORING_SLOPE else (1, 140)),
        full_marks_at=40, hard_cap=140, progress_key="pv")
    assert p_value_length is not None, "p-value suffix did not flip all strings. Raise hard_cap / pool size."
    STATE.pop("pv_starts", None)
    STATE.pop("pv_partial", None) # transient progress -> drop
    checkpoint(pvalue_suffix=YOUR_PVALUE_SUFFIX_LIST)  # checkpoint the finished phase

p_value_results = [p_value_query(string + YOUR_PVALUE_SUFFIX_LIST) for string in p_value_strings]   # cached -> free on re-run
log(f"\nP-VALUE SUFFIX LENGTH = {p_value_length}   per-string p-values: {[round(value, 4) for value in p_value_results]}")
log(f"all >= 0.01: {all(value >= 0.01 for value in p_value_results)} | target <=41: {p_value_length <= 41} | full marks <=40: {p_value_length <= 40}")
print_quota_usage()
print_known_pairs()


resumed p-value suffix from checkpoint (length 40)

P-VALUE SUFFIX LENGTH = 40   per-string p-values: [0.0107, 0.0107, 0.0107, 0.0107, 0.0107, 0.0107, 0.0107]
all >= 0.01: True | target <=41: True | full marks <=40: True
this session: 0 new p-value + 0 new bool queries sent 7 served from cache (1532 total stored).
  known pvalue pairs: 409 red, 404 green
  known bool pairs: 199 red, 210 green


### Solve the boolean endpoint


In [ ]:
# Solve the boolean endpoint
# Balanced-base method (see helpers): a 50-distinct-pair base B with exactly 17 reds lets B+[a,b] test any edge in one query. We greedily build a red chain whose every internal pair is red. Appended to a string it flips it once the chain is long enough.

# At green excess 70 the minimal lengths are 42 (some boundary pair green) or 40 (ALL 7 boundary pairs (end_i,c1) red). BOOL_BOUNDARY_OPTIMIZATION controls whether we chase 40: False -> stop at the safe minimal 42 (29 pts), fewest queries. True -> after the 42, reuse its chain as a fixed red body and search for a first token c1 red after ALL 7 string ends, giving [c1]+body = 40 tokens (<=41 -> 30 pts). Green-connected ends are verified via the base. The rest are confirmed with 7 real string checks per candidate. Reliably reaches 40 at G=70 for a few hundred extra queries. The safe 42 is banked first, so the search can never lose it.
BOOL_BOUNDARY_OPTIMIZATION = True
BOOL_K = 3 # plain start tokens for the 42 body
BOOL_TARGET_40_CAP = 2500 # max NEW bool queries for the c1 search

log("==== boolean endpoint ====", echo=False)
bool_strings = [tokens_to_array(token_entry["tokens"]).tolist() for token_entry in bool_token_ids]

# fully resume only if the phase is finished: a <=41 suffix, or boundary optimization already ran (bool_opt_done), or boundary optimization is off and a suffix exists.
phase_done = ("bool_suffix" in STATE and (len(STATE["bool_suffix"]) <= 41 or STATE.get("bool_opt_done") or not BOOL_BOUNDARY_OPTIMIZATION))
if phase_done:
    YOUR_BOOL_SUFFIX_LIST = STATE["bool_suffix"]
    bool_length = len(YOUR_BOOL_SUFFIX_LIST)
    log(f"resumed bool suffix from checkpoint (length {bool_length})")
else:
    # (re)build the balanced base + green-connected candidate pool
    if "bool_base" in STATE:
        bool_is_red, bool_candidate_pool, bool_base = make_bool_oracle(
            base=STATE["bool_base"], green_connected_seed=STATE.get("bool_candidates", []), n_candidates=160)
        log(f"resumed balanced base + {len(bool_candidate_pool)} green-connected candidates from checkpoint")
    else:
        bool_is_red, bool_candidate_pool, bool_base = make_bool_oracle(n_candidates=160)
    log(f"balanced base end: {bool_base[-1]} | green-connected candidates: {len(bool_candidate_pool)}")

    # (a) minimal no opt red chain -> the safe 42 (banked immediately as the fallback).
    if "bool_suffix" in STATE: # 42 already built on a prior run
        YOUR_BOOL_SUFFIX_LIST = STATE["bool_suffix"]
        bool_length = len(YOUR_BOOL_SUFFIX_LIST)
        log(f"resumed no-opt bool body (length {bool_length})")
    else:
        YOUR_BOOL_SUFFIX_LIST, bool_length = best_suffix(
            bool_is_red, bool_candidate_pool, bool_strings, bool_query, lambda bool_value: bool_value == 0, starts=bool_candidate_pool[:BOOL_K],
            check_lengths=grading_check_lengths("bool"),
            trim_range=(scoring_slope("bool") if TRIM_ONLY_IN_SCORING_SLOPE else (1, 140)),
            full_marks_at=41, hard_cap=140, progress_key="bool")
        assert bool_length is not None, "bool suffix did not flip all strings, raise n_candidates."
        STATE.pop("bool_starts", None)
        STATE.pop("bool_partial", None) # transient -> drop
        checkpoint(bool_suffix=YOUR_BOOL_SUFFIX_LIST)  # bank the safe 42 right away
        log(f"no-opt bool suffix length = {bool_length} (safe fallback, banked)")

    # (b) boundary optimisation: reuse the chain as a fixed red body and find a first token c1 red after all 7 string ends -> a 40-token suffix [c1]+body.
    if BOOL_BOUNDARY_OPTIMIZATION and bool_length >= 42:
        body = list(YOUR_BOOL_SUFFIX_LIST)[1:40]
        body_start = body[0]
        string_end_tokens = [string[-1] for string in bool_strings]
        green_connected_ends = [end_token for end_token in string_end_tokens if is_green_connected(bool_base, end_token)]
        log(f"boundary-opt ON: {len(green_connected_ends)}/7 green-connected ends (verifiable, rest via real strings)")
        candidate_pool = list(bool_candidate_pool)
        seen_tokens = set(candidate_pool) | set(bool_base) | set(body)
        extra_candidate_pool = (candidate_token for candidate_token in FREE if candidate_token not in seen_tokens)
        queries_before_search = api_stats["new_bool"]
        found_suffix = None
        candidate_index = 0
        while found_suffix is None and (api_stats["new_bool"] - queries_before_search) < BOOL_TARGET_40_CAP:
            if candidate_index >= len(candidate_pool): # grow the green-connected pool on demand
                added_candidate = False
                for candidate_token in extra_candidate_pool:
                    if is_green_connected(bool_base, candidate_token):
                        candidate_pool.append(candidate_token)
                        added_candidate = True
                        break
                if not added_candidate:
                    break
            first_token = candidate_pool[candidate_index]
            candidate_index += 1
            if first_token in body or first_token == body_start:
                continue
            if not all(bool_is_red(end_token, first_token) for end_token in green_connected_ends): # red after every verifiable end
                continue
            if not bool_is_red(first_token, body_start): # connects (red) to the body
                continue
            candidate = [first_token] + body # 40 tokens, all internal pairs red
            if all(bool_query(string + candidate) == 0 for string in bool_strings): # confirm all 7 flip at length 40
                found_suffix = candidate
        if found_suffix is not None:
            YOUR_BOOL_SUFFIX_LIST, bool_length = found_suffix, 40
            log("boundary-opt FOUND a 40-token suffix (full marks <=41).")
        else:
            log(f"boundary-opt: no 40 within {BOOL_TARGET_40_CAP} new queries, keeping {bool_length}.")

    checkpoint(bool_suffix=YOUR_BOOL_SUFFIX_LIST, bool_opt_done=True) # phase finished

bool_results = [bool_query(string + YOUR_BOOL_SUFFIX_LIST) for string in bool_strings] # cached -> free on re-run
log(f"\nBOOL SUFFIX LENGTH = {bool_length}   per-string bool: {bool_results}")
log(f"all 0: {all(bool_value == 0 for bool_value in bool_results)} | target <=42: {bool_length <= 42} | full marks <=41: {bool_length <= 41}")
print_quota_usage()
print_known_pairs()


resumed bool suffix from checkpoint (length 40)

BOOL SUFFIX LENGTH = 40   per-string bool: [0, 0, 0, 0, 0, 0, 0]
all 0: True | target <=42: True | full marks <=41: True
this session: 0 new p-value + 7 new bool queries sent 7 served from cache (1539 total stored).
  known pvalue pairs: 409 red, 404 green
  known bool pairs: 199 red, 210 green


# Export your solution


Set your `STUDENT_ID` below. The files are written to a local
`llm_assignment_3_submission-<STUDENT_ID>/` folder — zip that folder for Moodle.


In [ ]:
# Local submission folder (replaces the Colab/Drive + git-clone template).
import re
from pathlib import Path

STUDENT_ID = "25-936-154"
assert re.match(r"^\d{2}-\d{3}-\d{3}$", STUDENT_ID), "Student ID should have the format 'dd-ddd-ddd'"

SOLUTIONS_PATH = Path(f"./llm_assignment_3_submission-{STUDENT_ID}")
SOLUTIONS_PATH.mkdir(parents=True, exist_ok=True)
log(f"saving solution files to: {SOLUTIONS_PATH.resolve()}")


saving solution files to: /Users/gorkemkadirsolun/Library/CloudStorage/GoogleDrive-gorkemkadirsolun@gmail.com/My Drive/Ders_Course/Large_Language_Models/large-language-models-homeworks/third/question2/llm_assignment_3_submission-25-936-154


In [19]:
# Create Q2 key.txt with your API key
with open(SOLUTIONS_PATH / "Q2_key.txt", "w") as f:
  f.write(API_KEY)

In [20]:
# Suffixes discovered by the attack cells above.
YOUR_PVALUE_SUFFIX = np.array(YOUR_PVALUE_SUFFIX_LIST, dtype=int)
YOUR_BOOL_SUFFIX   = np.array(YOUR_BOOL_SUFFIX_LIST,   dtype=int)

assert isinstance(YOUR_PVALUE_SUFFIX, np.ndarray) and isinstance(YOUR_PVALUE_SUFFIX[0], np.int64)
assert isinstance(YOUR_BOOL_SUFFIX,   np.ndarray) and isinstance(YOUR_BOOL_SUFFIX[0], np.int64)
assert len(YOUR_PVALUE_SUFFIX) <= 140 and len(YOUR_BOOL_SUFFIX) <= 140
for endpoint_name, suffix_array in [("pvalue", YOUR_PVALUE_SUFFIX), ("bool", YOUR_BOOL_SUFFIX)]:
    assert all(0 <= int(token) < 50277 for token in suffix_array), f"{endpoint_name} suffix has a token id outside [0, 50277)"
log(f"pvalue suffix length: {len(YOUR_PVALUE_SUFFIX)} | bool suffix length: {len(YOUR_BOOL_SUFFIX)}")
log(f"token-id range OK: pvalue [{int(YOUR_PVALUE_SUFFIX.min())}, {int(YOUR_PVALUE_SUFFIX.max())}], "
    f"bool [{int(YOUR_BOOL_SUFFIX.min())}, {int(YOUR_BOOL_SUFFIX.max())}]")


pvalue suffix length: 40 | bool suffix length: 40
token-id range OK: pvalue [1879, 50132], bool [1595, 48046]


### Validate your suffixes (optional)

Run this cell to check that your suffixes pass all 7 strings before exporting. This uses **14 API calls** (7 for pvalue + 7 for bool).


In [21]:
# Validate your suffixes against all 7 strings for each endpoint. Uses the cached p_value_query()/bool_query() wrappers, so on a re-run (and even on the first run, since the solver cells already queried these exact string+suffix combinations) this costs zero new API queries.
p_value_strings_check = [tokens_to_array(token_entry["tokens"]).tolist() for token_entry in pvalue_token_ids]
bool_strings_check    = [tokens_to_array(token_entry["tokens"]).tolist() for token_entry in bool_token_ids]

log("=== p-value suffix vs all 7 strings (need p >= 0.01) ===")
for index, string in enumerate(p_value_strings_check):
    pvalue = p_value_query(string + [int(token) for token in YOUR_PVALUE_SUFFIX])
    log(f"  string {index + 1}/7: p = {pvalue:.4f}   {'OK' if pvalue >= 0.01 else 'STILL WATERMARKED'}")

log("\n=== bool suffix vs all 7 strings (need bool == 0) ===")
for index, string in enumerate(bool_strings_check):
    bool_value = bool_query(string + [int(token) for token in YOUR_BOOL_SUFFIX])
    log(f"  string {index + 1}/7: bool = {bool_value}   {'OK' if bool_value == 0 else 'STILL WATERMARKED'}")

print_quota_usage()


=== p-value suffix vs all 7 strings (need p >= 0.01) ===
  string 1/7: p = 0.0107   OK
  string 2/7: p = 0.0107   OK
  string 3/7: p = 0.0107   OK
  string 4/7: p = 0.0107   OK
  string 5/7: p = 0.0107   OK
  string 6/7: p = 0.0107   OK
  string 7/7: p = 0.0107   OK

=== bool suffix vs all 7 strings (need bool == 0) ===
  string 1/7: bool = 0   OK
  string 2/7: bool = 0   OK
  string 3/7: bool = 0   OK
  string 4/7: bool = 0   OK
  string 5/7: bool = 0   OK
  string 6/7: bool = 0   OK
  string 7/7: bool = 0   OK
this session: 0 new p-value + 7 new bool queries sent 21 served from cache (1539 total stored).


### Export your suffixes


In [22]:
with open(SOLUTIONS_PATH / 'Q2_pvalue.npy', 'wb') as f:
    np.save(f, YOUR_PVALUE_SUFFIX)

with open(SOLUTIONS_PATH / 'Q2_bool.npy', 'wb') as f:
    np.save(f, YOUR_BOOL_SUFFIX)

print("Saved Q2_pvalue.npy and Q2_bool.npy")

Saved Q2_pvalue.npy and Q2_bool.npy
